# P35 — FlashAttention: atención exacta, rápida y eficiente en memoria, consciente de la E/S

## 1. Título y paper

**Paper:** *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*  
**Autoría:** Tri Dao, Daniel Y. Fu, Stefano Ermon, Atri Rudra, Christopher Ré  
**Año y venue:** 2022 · arXiv:2205.14135 · NeurIPS 2022  
**Nivel:** L4 · **Motor:** `flashattention`  
**Ficha completa:** [`P35_flashattention`](../../papers/foundational/P35_flashattention/README.md)

**Hito:** El cuello de botella de la atención no eran los FLOPs sino las lecturas y escrituras a memoria. Y la solución es EXACTA, no aproximada.

- [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Durante años se atacó el coste O(n²) de la atención con aproximaciones (dispersa, lineal), que perdían calidad y a menudo ni siquiera eran más rápidas en la práctica.
2. Ejecutar una implementación mínima de la propuesta: Reorganizar el cálculo por bloques que caben en la memoria rápida del chip, evitando materializar la matriz de atención completa en la memoria lenta.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P08
- P34


## 4. Intuición

El cálculo de la atención no es lento por hacer muchas cuentas: es lento por escribir y leer una matriz enorme en la memoria lenta de la GPU. La solución no es calcular menos, sino no escribirla nunca.


## 5. Concepto mínimo

```text
Estándar:  calcular S = QKᵀ  → ESCRIBIR n×n en HBM → leer → softmax → escribir → leer → ×V
Flash   :  recorrer por bloques que caben en SRAM, con softmax incremental reescalado

    mismos FLOPs · mismo resultado EXACTO · muchísimos menos accesos a memoria
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('flashattention', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cambian los FLOPs entre ambas versiones?
2. ¿Y el resultado numérico?
3. ¿Qué crece más rápido con n: el cómputo o la memoria que hay que mover?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('flashattention', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('flashattention', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Los FLOPs son idénticos y el resultado es **exacto**: no es una aproximación. Lo que cambia es cuántos elementos viajan entre la memoria rápida del chip y la lenta. Ese era el cuello de botella real, y durante años se atacó el equivocado.


## 10. Comentario pedagógico

Es una lección más general que la atención: en hardware moderno, **mover datos cuesta más que calcular**. Muchas optimizaciones «obvias» de FLOPs no aceleran nada porque el proceso está limitado por memoria. Conecta directamente con el modelo roofline de la clase 081.


## 11. Error o anti-patrón deliberado

Anti-patrón: optimizar FLOPs sin mirar el tráfico de memoria.


In [ ]:
print('Reducir FLOPs con atencion aproximada fue el enfoque dominante 2019-2021.')
print('Muchos de esos metodos NO eran mas rapidos en la practica:')
print('  bajaban el computo pero seguian materializando matrices en memoria lenta.')

## 12. Corrección

El criterio correcto es contar accesos a memoria, no operaciones:


In [ ]:
d, M = 64, 100_000
for n in (1024, 16384):
    print(f'n={n:>6} · FLOPs={2*n*n*d:>15,} · HBM estandar={2*n*n+2*n*d:>13,} '
          f'· HBM flash={int(4*n*d*(n*d/M)):>12,}')

## 13. Desafío guiado

Calcula a partir de qué n la matriz de atención deja de caber en 40 GB, y compáralo con el contexto que anuncian los modelos actuales.


In [ ]:
r = run_paper_lab('flashattention', seed=3)['result']
show(r)

## 14. Desafío autónomo

Perfila una implementación de atención en tu GPU con y sin la versión optimizada de tu biblioteca, a varias longitudes. Reporta tiempo y memoria máxima, no solo tiempo.


## 15. Evidencia de aprendizaje

Guarda la tabla de FLOPs frente a accesos a memoria y tu explicación de por qué el algoritmo es exacto y aun así mucho más rápido.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P35_flashattention/README.md) · evaluación formal: [`assessments/papers/P35_flashattention.md`](../../assessments/papers/P35_flashattention.md)


## 16. Cierre

Ya cabe el contexto largo. La pregunta siguiente es incómoda: ¿lo usa el modelo?


## 17. Conexión con el siguiente hito

- contexto largo en modelos de producción

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
